In [ ]:
# input text that the model will be trained on
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
# all unique characters
chars = sorted((list(set(text))))
vocab_size = len(chars)

In [ ]:
# create mapping form characters to integers
# Other mappings: google uses sentencepiece, openai uses tiktoken
# Balance between length of integers with length of vocab
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]   ## take string, output list of integers
decode = lambda l: ''.join([itos[i] for i in l])  ## take list of integers, output a string

In [ ]:
# now can encode dataset and into a tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
# split into train and validation sets
# helps understand whether its overfitting
n = int(0.9*len(data))
train_data = data[:n]  ## first 90 percent
val_data = data[n:]

In [ ]:
# size of input
block_size = 8
train_data[:block_size+1]

In [ ]:
# reducing block size means efficienct and for model to be familiar with any block size input up to "block_size"
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} that target: {target}")

In [ ]:
# mini batches of blocks since GPUs are very good as parallel processing
torch.manual_seed(1337)
batch_size = 4  ## how many independent sequences will be processed in parallel
block_size = 8  ## what is the maximum context length for predictions

def get_batch(split):
    # generate a small batch
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  ## random integers of size batch_size
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):  ## batch dimension
    for t in range(block_size):  ## time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # a lookup table of shape vocab_size*vocab_size
        # each token id maps to a vector os size vocab_size
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # each token id in the (batch_size, block_size) tensor is replaced by a new dimension:
        # list of integers of length vocab_size representing the embedding
        logits = self.token_embedding_table(idx)
        
        if targets is None:
            loss = None
        else:
            # reshape to work with cross_entropy function expectations
            B, T, C = logits.shape  ## batch_size, block_size, vocab size
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  ## computes loss between predictions and targets

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (batch size by token size) array of indices
        # For each batch, u'd get the new tokens; generates predictions for all batches at once
        for _ in range(max_new_tokens):
            logits, loss = self(idx)  ## get predictions for all positions
            logits = logits[:, -1, :]  ## keep only last postiion's predictions (next token) for each batch
            probs = F.softmax(logits, dim=-1)  ## make em sum to 1 to get probabilities
            idx_next = torch.multinomial(probs, num_samples=1)  ## randomly sample on token from distribution
            idx = torch.cat((idx, idx_next), dim=1)  ## append sampled index to the running sequence
        return idx

m = BigramLanguageModel(vocab_size)
out = m(xb, yb)

## Sample generation
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))